# Manipulação de dados

Tal como referido no tutorial sobre [análise de dados](./03-pandas-analysis.ipynb), a biblioteca [pandas](https://pandas.pydata.org/) oferece uma vasta gama de métodos e operações para análise e manipulação de dados a vários níveis. Neste tutorial cobriremos mais uma pequena parte da funcionalidade disponibilizada pela biblioteca, com foco no processamento e transformação de dados. 

Para isso, vamos começar por importar a biblioteca: 

In [ ]:
import pandas as pd

## Conjunto de dados

Vamos usar novamente o conjunto de dados com informação sobre filmes extraída da [IMDb](https://www.imdb.com/).

### Carregamento dos dados

In [ ]:
import os

data_path = '../data/' if os.path.exists('../data/') else 'https://raw.githubusercontent.com/HLT-ISCTE/DECD/main/data/'

movies_df = pd.read_csv(data_path + 'IMDB-Movie-Data.csv', index_col='Title')

### Visualização

Vamos começar por relembrar o conteúdo deste conjunto de dados:

In [ ]:
movies_df

E as suas características:

In [ ]:
movies_df.info()

### Limpeza de colunas

Além disso, vamos aplicar a mesma limpeza às colunas que anteriormente:

In [ ]:
movies_df.columns = [
    'rank', 'genre', 'description', 
    'director', 'actors', 'year', 
    'runtime', 'rating', 'votes', 
    'revenue_millions', 'metascore'
]
movies_df.columns

In [ ]:
movies_df.drop(columns=['rank', 'description', 'actors'], inplace=True)

In [ ]:
movies_df.head(3)

## Valores em falta

Já vimos anteriormente que existem **128** valores em falta na coluna `revenue_millions` e **64** valores em falta na coluna `metascore`:

In [ ]:
movies_df.isnull().sum()

As abordagens mais comuns para lidar com valores em falta são:

1. Apagar as linhas ou colunas com valores em falta
2. Prencher os valores em falta com valores obtidos usando uma técnica chamada imputação

#### Remoção de valores em falta

Cientistas e analistas de dados são regularmente confrontados com o dilema de descartar ou imputar valores em falta. Esta é uma decisão que requer conhecimento dos dados e do contexto em que eles são utilizados. No entanto, normalmente, só é recomendada a remoção das entradas com valores em falta quando estas representam uma porção insignificante do conjunto de dados total. Pelo contrário, só é recomendada a remoção de atributos com valores em falta quando estes estão em maioria. 

O método `dropna` pode ser usado para descartar valores em falta de uma tabela de dados. O comportamento predefinido é apagar todas as **entradas** que têm pelo menos um valor em falta:

In [ ]:
movies_df.dropna()

No caso deste conjunto de dados, seriam removidas `1000 - 838 = 162` entradas ao aplicar esta operação. Isto é um desperdício, uma vez que existem dados que podem ser importantes nas outras colunas dessas entradas.

Para descartar atributos com valores em falta podemos usar o método `dropna` com o argumento `axis='columns'` ou `axis=1`:

In [ ]:
movies_df.dropna(axis='columns')

Neste caso, são mantidas as 1000 entradas, mas as colunas `revenue_millions` e `metascore` são removidas.


#### Imputação

Imputação (o processo de substituir valores em falta por valores representativos) é uma técnica convencional de engenharia de atributos usada para evitar descartar entradas com valores em falta. Na prática, consiste em substituir os valores em falta por valores representativos, como por exemplo a média ou a mediana do atributo no conjunto de dados.

Como exemplo, vamos usar esta estratégia para lidar com os valores em falta na coluna `revenue_millions`:

In [ ]:
revenue = movies_df['revenue_millions']
revenue_mean = revenue.mean()
revenue_mean

Agora que já temos a média, podemos usar o método `fillna` para preencher os valores em falta com esse valor:

In [ ]:
revenue.fillna(revenue_mean, inplace=True)

**Nota**: O método foi chamado sobre a série `revenue` pois só queremos preencher os valores em falta para esse atributo e não todos os valores em falta na tabela de dados.

In [ ]:
movies_df.isnull().sum()

**Nota**: Imputar uma todos os valores em falta de uma coluna com o mesmo valor é um exemplo básico da aplicação técnica. Uma ideia melhor seria usar uma imputação mais granular. Por exemplo, podiamos calcular a média para cada género de filme ou para cada realizador e usar esses valores para preencher os valores em falta em entradas com as mesmas características.

## Transformação de atributos

Existem múltiplas razões que levam à necessidade de transformar os atributos de conjunto de dados de alguma forma. Por exemplo:

- A representação de um determinado atributo não é a mais adequada no contexto do problema que queremos abordar
- As abordagens de extração de conhecimento que queremos aplicar não são compatíveis com um determinado tipo de atributo
- Queremos agrupar atributos ou gerar novos atributos com base nos existentes

Como exemplo, vamos transformar o atributo `rating` num atributo categórico que tem o valor `'good'` se a classificação for igual ou superior a 8.0 e o valor `'bad'` caso contrário. Para isso, vamos começar por criar a função que faz essa transformação:

In [ ]:
def rating_function(x):
    if x >= 8.0:
        return 'good'
    else:
        return 'bad'

É possível iterar sobre uma série ou tabela de dados da mesma forma que sobre uma lista e usar essa abordagem para aplicar a função a todas as entradas:

In [ ]:
pd.Series([rating_function(r) for r in movies_df['rating']], index=movies_df.index)

No entanto, essa é uma operação que se torna lenta em conjuntos de dados de grande dimensão. Uma alternativa mais eficiente é usar o método `apply`:

In [ ]:
movies_df['rating'].apply(rating_function)

**Nota**: O método `apply` é mais eficiente pois usa vetorização, isto é, a função é aplicada a todas as entradas de uma só vez.

Para adicionar o novo atributo ou substituir o existente, podemos usar a analogia da tabela de dados como um dicionário:

In [ ]:
movies_df['rating_category'] = movies_df['rating'].apply(rating_function)
movies_df.head(2)

**Nota**: Muitas vezes é útil usar uma função anónima como argumento do método `apply`. Estas têm a sintaxe `lambda <argumentos>: <expressão>`. Por exemplo, a transformação anterior também poderia ser feita da seguinte forma: 

In [ ]:
movies_df['rating'].apply(lambda x: 'good' if x >= 8.0 else 'bad')

Se olharmos para a informação dada pelo método `info` podemos ver que atributo que criámos tem o tipo `object`, tal como os outros atributos que são cadeias de caracteres:

In [ ]:
movies_df.info()

Podemos explicitar que se trata de um atributo categórico usando o método `astype`:

In [ ]:
movies_df['rating_category'] = movies_df['rating_category'].astype('category')
movies_df.info()

A função `factorize` pode ser usada para transformar os valores dum atributo categórico em valores inteiros:

In [ ]:
pd.factorize(movies_df['rating_category'])

Para exemplificar a criação de um novo atributo a partir de uma combinação dos existentes, vamos gerar um novo atributo que diz a variação entre as duas classificações de um filme (`rating` e `metascore`):

In [ ]:
# We divide the metascore by 10 so that both ratings are in the same scale
movies_df['rating_difference'] = movies_df.apply(lambda x: x['rating'] - x['metascore'] / 10, axis='columns')  
movies_df.head()

Neste caso, estamos a aplicar o método `apply` sobre a tabela de dados e a explicitar o argumento `axis='columns'` de forma a ter acesso a todos os atributos e podermos usá-los na geração do novo atributo. 

**Nota**: Como as séries são construídas em cima de arrays *NumPy*, este atributo também pode ser obtido fazendo as operações diretamente sobre os dois atributos:

In [ ]:
movies_df['rating'] - movies_df['metascore'] / 10

**Nota**: Uma área em que o método `apply` é usado exaustivamente é o processamento de língua natural. Nesse contexto é necessário aplicar uma panóplia de funções de limpeza e manipulação de texto para preparar os dados para aplicação de abordagens de aprendizagem automática.

## Considerações finais

A capacidade de analisar, explorar, transformar e visualizar dados é essencial na ciência de dados. Os vários passos deste processo ocupam uma grande parte do tempo de quem trabalha nesta área. Como tal, é importante que seja possível reproduzir de forma fácil o processo de análise e manipulação feito sobre um determinado conjunto de dados e/ou que, pelo menos, o seu resultado seja guardado. 

Tal como para a leitura de conjuntos de dados em vários formatos, a biblioteca *pandas* fornece um conjunto de métodos para guardar conjuntos de dados nesses mesmos formatos. Por exemplo, podemos usar o método `to_csv` para guardar o nosso conjunto de dados processado num ficheiro CSV:

In [ ]:
movies_df.to_csv('IMDB-Movie-Data-Processed.csv')

Tal como referido anteriormente, a biblioteca *pandas* oferece uma vasta gama de métodos e operações para análise e manipulação de dados a vários níveis. Neste tutorial cobrimos apenas uma pequena parte da funcionalidade disponibilizada pela biblioteca: a funcionalidade básica que é usada em quase todas as tarefas de análise e manipulação de dados. Para explorar alguns temas mais a fundo, recomendamos os [tutoriais da biblioteca pandas](https://pandas.pydata.org/pandas-docs/stable/tutorials.html) e o [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/).  